# QLoRA fine-tuning for maintenance-fault triage

Runs the full experiment from the repository's `maintriage` package on a Colab GPU (a free T4 is enough): classical baselines, zero-shot Qwen2.5-1.5B-Instruct, QLoRA fine-tuning, and evaluation on the same held-out test split.

All logic lives in `src/maintriage/`; this notebook only installs it, runs it and shows the results. **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Install
Clones the repository and installs the package with its GPU extras.

In [ ]:
!git clone -q https://github.com/dojedaro/qlora-maintenance-triage.git
%cd qlora-maintenance-triage
!pip install -q -e ".[gpu]"

## Run the experiment
About 20–30 minutes on a T4. Writes `metrics.json`, per-class scores and every test prediction to `results/`.

In [ ]:
!python -m maintriage.run --out results

## Results

In [ ]:
import json

import pandas as pd

metrics = json.load(open("results/metrics.json"))
print(json.dumps({k: metrics[k] for k in ("dataset", "training", "environment")}, indent=2))
pd.DataFrame(metrics["models"]).T

Where the fine-tuned model and the classical baseline disagree:

In [ ]:
preds = pd.read_csv("results/test_predictions.csv")
disagree = preds[preds.qlora != preds.tfidf_logreg]
print(f"{len(disagree)} of {len(preds)} test reports")
print("QLoRA right:", (disagree.qlora == disagree.gold).sum(), "| TF-IDF right:", (disagree.tfidf_logreg == disagree.gold).sum())
disagree[["fault", "gold", "qlora", "tfidf_logreg"]].head(15)

## Download the results
Everything except the adapter weights, which are large and easy to regenerate.

In [ ]:
!cd results && zip -q -r ../results.zip . -x 'adapter/*'
from google.colab import files

files.download("results.zip")